# **Model Evaluation**
Here we will evaluate the given, optimised XGBoost model more closely to get additional information and insights.
We will use the results of our HyperParameter selection and try to assure that our models meets our requirements.

### Requirements:
- Perform better than RandomGuessing
- Perform better than ZeroGuessing
- Respond to a new example in less than one second 
- Model should at least identify 75% fraud payments (Recall >= 0.75)
- Model should make min 15% accurate calls (Precision >15)


In [2]:
import sys
from pathlib import Path
import pandas as pd
import time
import random
import statistics
import joblib
from sklearn.pipeline import Pipeline
sys.path.append('..')  
from src.models.random_guessing import RandomGuessing
from src.models.zero_guessing import ZeroGuessing
from src.evaluation.evaluation_metrics import EvaluationMetrics

from src.data.loader import Loader
from src.data.splitter import split_data

loader = Loader().load()
X_train, X_test, y_train, y_test= split_data(loader.df)

0.8966472893870382
0


## **Setup Model**
In this section we will import our pipeline with feature engineering and the given model.

In [3]:
model_path = Path.cwd().parent / "src" / "models" / "xgb" / "pipeline.pkl"
pipeline:Pipeline = joblib.load(filename=model_path)

## **Benchmarks (Random and Zero guessing)**
In this section we will compare the models performance against our two Base Models.


We expect the model to perform significantly better than both the Base Models.

We compare the models based on their ROC-AUC score

In [4]:
model = ZeroGuessing()
y_pred_test = model.predict_many(X=X_test)
zero_guessing_metrics = EvaluationMetrics(y=y_test, y_pred=y_pred_test)
zero_guessing_metrics.roc_auc

/Users/tomseidel/Desktop/Learning/ML/fraud-detection/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


0.5

In [5]:
model = RandomGuessing()
y_pred_test = model.predict_many(X=X_test)
random_guessing_metrics = EvaluationMetrics(y=y_test, y_pred=y_pred_test)
random_guessing_metrics.roc_auc

0.49725702679819395

In [6]:
y_pred_test = pipeline.predict_proba(X_test)[:, 1]
xgb_guessing_metrics = EvaluationMetrics(y=y_test, y_pred=y_pred_test)
xgb_guessing_metrics.roc_auc

0.919471123971491

In [7]:
score_dataframe = pd.DataFrame([
    {"model": "Zero Guessing",   "roc_auc": zero_guessing_metrics.roc_auc},
    {"model": "Random Guessing", "roc_auc": random_guessing_metrics.roc_auc},
    {"model": "XGBoost",         "roc_auc": xgb_guessing_metrics.roc_auc},
])
score_dataframe[score_dataframe["model"].isin(["Zero Guessing", "Random Guessing", "XGBoost"])]

,model,roc_auc
0,Zero Guessing,0.500000
1,Random Guessing,0.497257
2,XGBoost,0.919471


as expected, we can see that our optimised model performs way better than both our Benchmarks.

## **Time Measurement**
In this section we will check the response time of our trained model. We expect and request it to respond in less than one second.

We will do this by predict 100 examples sequencially and then get mean, min and max duration

In [8]:
durations = []

for i in range(100):
    start_time = time.time()
    sample = X_test.iloc[[random.randint(0, len(X_test)-1)]]  
    pipeline.predict_proba(sample)[:, 1]
    end_time = time.time()
    durations.append(end_time - start_time)
    
print("Maximum:",round(max(durations),2),"s")
print("Minimum:",round(min(durations),2),"s")
print("Average:",round(statistics.mean(durations),2),"s")

Maximum: 0.05 s
Minimum: 0.02 s
Average: 0.02 s


The model is running faster than expected and meets our requirements.

## **Precision and Recall**
In this section we want to make sure, that our model meets our given requirements for Recall and Precision which we defined at the top.

We are using the optimised Threshold we found in Threshold Tuning (0.075)

In [12]:
y_pred_test = pipeline.predict_proba(X_test)[:, 1]
xgb_guessing_metrics = EvaluationMetrics(y=y_test, y_pred=y_pred_test, threshold=0.3)
xgb_guessing_metrics.print_eval_report(headline="XGBmodel")



XGBmodel
- Precision: 0.1573254221816522
- Recall: 0.8481791338582677
- F1: 0.2654192654192654
- ROC-AUC: 0.919471123971491


The model meets our requirements.

## Conclusion

The model meets all four requirements defined at the top of this notebook: it substantially outperforms both baselines (ROC-AUC 0.92 vs. 0.50), responds in under 50ms per sample on average, achieves Recall ≥ 0.75, and Precision > 15%.

The decision threshold of 0.3 (tuned separately in `src/evaluation/treshold.py`) is the key lever between precision and recall. Lowering the threshold catches more fraud but increases false positives; the chosen value reflects the priority of recall in a fraud detection context